# **Breast Cancer Wisconsin (Diagnostic) Data Analysis**

This notebook analyzes the Breast Cancer Wisconsin (Diagnostic) dataset from `data.ods`.

# Step 1: Installing Required Packages

In [ ]:
!pip install odfpy

Defaulting to user installation because normal site-packages is not writeable


# Step 2:Loading the ODS Dataset and Printing Sheet Name & Headers

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

filename = 'data.ods'
doc = load(filename)

# Extract the first sheet
sheets = doc.getElementsByType(Table)
sheet = sheets[0]
sheet_name = sheet.getAttribute('name')
rows = sheet.getElementsByType(TableRow)

# Extract headers
header_row = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
]
print(f'Sheet Name: {sheet_name}')
print(f'Headers: {headers}')

Sheet Name: data
Headers: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


# Step 3: Displaying Sample Rows

In [ ]:
print("Sample rows:")
for i, row in enumerate(rows[1:6], 1):
    cells = row.getElementsByType(TableCell)
    values = [
        teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    print(f"Row {i}: {values}")

Sample rows:
Row 1: ['842302', 'M', '17.99', '10.38', '122.8', '1001', '0.1184', '0.2776', '0.3001', '0.1471', '0.2419', '0.07871', '1.095', '0.9053', '8.589', '153.4', '0.006399', '0.04904', '0.05373', '0.01587', '0.03003', '0.006193', '25.38', '17.33', '184.6', '2019', '0.1622', '0.6656', '0.7119', '0.2654', '0.4601', '0.1189']
Row 2: ['842517', 'M', '20.57', '17.77', '132.9', '1326', '0.08474', '0.07864', '0.0869', '0.07017', '0.1812', '0.05667', '0.5435', '0.7339', '3.398', '74.08', '0.005225', '0.01308', '0.0186', '0.0134', '0.01389', '0.003532', '24.99', '23.41', '158.8', '1956', '0.1238', '0.1866', '0.2416', '0.186', '0.275', '0.08902']
Row 3: ['84300903', 'M', '19.69', '21.25', '130', '1203', '0.1096', '0.1599', '0.1974', '0.1279', '0.2069', '0.05999', '0.7456', '0.7869', '4.585', '94.03', '0.00615', '0.04006', '0.03832', '0.02058', '0.0225', '0.004571', '23.57', '25.53', '152.5', '1709', '0.1444', '0.4245', '0.4504', '0.243', '0.3613', '0.08758']
Row 4: ['84348301', 'M', '11.4

# Step 4: Cleaning the Dataset

In [ ]:
# Standardize headers: strip, lower-case, replace spaces with underscores
clean_headers = [h.strip().lower().replace(' ', '_') for h in headers]

clean_data = []
for row in rows[1:]:
    cells = row.getElementsByType(TableCell)
    if not cells or all(len(cell.getElementsByType(P)) == 0 for cell in cells):
        continue  # Skip empty rows
    values = [
        teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    # Pad values if row is short
    if len(values) < len(clean_headers):
        values += [''] * (len(clean_headers) - len(values))
    # Convert empty strings to None
    values = [v if v != '' else None for v in values]
    # Build dictionary
    record = dict(zip(clean_headers, values))
    clean_data.append(record)

print(f"Loaded and cleaned {len(clean_data)} records.")
print("Sample cleaned record:")
print(clean_data[0])

Loaded and cleaned 569 records.
Sample cleaned record:
{'id': '842302', 'diagnosis': 'M', 'radius_mean': '17.99', 'texture_mean': '10.38', 'perimeter_mean': '122.8', 'area_mean': '1001', 'smoothness_mean': '0.1184', 'compactness_mean': '0.2776', 'concavity_mean': '0.3001', 'concave_points_mean': '0.1471', 'symmetry_mean': '0.2419', 'fractal_dimension_mean': '0.07871', 'radius_se': '1.095', 'texture_se': '0.9053', 'perimeter_se': '8.589', 'area_se': '153.4', 'smoothness_se': '0.006399', 'compactness_se': '0.04904', 'concavity_se': '0.05373', 'concave_points_se': '0.01587', 'symmetry_se': '0.03003', 'fractal_dimension_se': '0.006193', 'radius_worst': '25.38', 'texture_worst': '17.33', 'perimeter_worst': '184.6', 'area_worst': '2019', 'smoothness_worst': '0.1622', 'compactness_worst': '0.6656', 'concavity_worst': '0.7119', 'concave_points_worst': '0.2654', 'symmetry_worst': '0.4601', 'fractal_dimension_worst': '0.1189'}


# Step 5:  Saving the Cleaned Dataset to ODS

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P

cleaned_file = 'cleaned_breast_cancer.ods'
doc_clean = OpenDocumentSpreadsheet()
table_clean = Table(name='CleanedBreastCancer')

# Header row
header_row = TableRow()
for h in clean_headers:
    cell = TableCell()
    cell.addElement(P(text=h))
    header_row.addElement(cell)
table_clean.addElement(header_row)

# Data rows
for rec in clean_data:
    row = TableRow()
    for h in clean_headers:
        val = rec.get(h, '')
        cell = TableCell()
        cell.addElement(P(text=str(val) if val is not None else ''))
        row.addElement(cell)
    table_clean.addElement(row)

doc_clean.spreadsheet.addElement(table_clean)
doc_clean.save(cleaned_file)
print(f"Cleaned dataset saved as: {cleaned_file}")

Cleaned dataset saved as: cleaned_breast_cancer.ods


# Step 6: Questions

## Q1: How many malignant and benign cases are there in the dataset?

In [ ]:
malignant = sum(1 for rec in clean_data if rec.get('diagnosis') == 'M')
benign = sum(1 for rec in clean_data if rec.get('diagnosis') == 'B')
print(f"Malignant: {malignant}")
print(f"Benign: {benign}")

Malignant: 212
Benign: 357


## Q2: What is the proportion of malignant cases in the dataset?

In [ ]:
total_cases = sum(1 for rec in clean_data if rec.get('diagnosis') in ['M', 'B'])
malignant_cases = sum(1 for rec in clean_data if rec.get('diagnosis') == 'M')

if total_cases > 0:
    proportion = malignant_cases / total_cases * 100
    print(f"Proportion of malignant cases: {proportion:.2f}%")
else:
    print("No valid diagnosis data found.")

Proportion of malignant cases: 37.26%


## Q3: What is the most common diagnosis in the dataset?

In [ ]:
diagnosis_counts = {'M': 0, 'B': 0}
for rec in clean_data:
    diag = rec.get('diagnosis')
    if diag in diagnosis_counts:
        diagnosis_counts[diag] += 1

if diagnosis_counts['M'] > diagnosis_counts['B']:
    print(f"Most common diagnosis: Malignant ({diagnosis_counts['M']} cases)")
elif diagnosis_counts['B'] > diagnosis_counts['M']:
    print(f"Most common diagnosis: Benign ({diagnosis_counts['B']} cases)")
elif diagnosis_counts['M'] == diagnosis_counts['B'] and diagnosis_counts['M'] > 0:
    print(f"Equal number of Malignant and Benign cases: {diagnosis_counts['M']} each")
else:
    print("No diagnosis data found.")

Most common diagnosis: Benign (357 cases)


# Step 7: Tasks

## Task 1: Feature with Highest Average Difference Between Malignant and Benign

In [ ]:
from odf.opendocument import load, OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype
import numpy as np

# Load cleaned data
filename = 'cleaned_breast_cancer.ods'
doc = load(filename)
sheet = doc.getElementsByType(Table)[0]
rows = sheet.getElementsByType(TableRow)

# Extract headers
header_row = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
]
data = []
for row in rows[1:]:
    cells = row.getElementsByType(TableCell)
    values = [
        teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    if len(values) < len(headers):
        values += [''] * (len(headers) - len(values))
    record = dict(zip(headers, values))
    data.append(record)

# Compute means for each feature by diagnosis
numeric_features = [h for h in headers if h not in ('id', 'diagnosis')]
summary = []
for feature in numeric_features:
    try:
        vals_m = [float(rec[feature]) for rec in data if rec['diagnosis'] == 'M' and rec[feature] not in ('', None)]
        vals_b = [float(rec[feature]) for rec in data if rec['diagnosis'] == 'B' and rec[feature] not in ('', None)]
        if vals_m and vals_b:
            mean_m = np.mean(vals_m)
            mean_b = np.mean(vals_b)
            diff = abs(mean_m - mean_b)
            summary.append((feature, mean_m, mean_b, diff))
    except Exception:
        continue
summary.sort(key=lambda x: x[3], reverse=True)
top_feature, top_m, top_b, top_diff = summary[0]

# Add new columns to each record (only for the top feature)
for rec in data:
    rec['mean_malignant_' + top_feature] = round(top_m, 3)
    rec['mean_benign_' + top_feature] = round(top_b, 3)
    rec['abs_difference_' + top_feature] = round(top_diff, 3)

# Save to ODS (one sheet, original data + new columns)
doc_out = OpenDocumentSpreadsheet()
table = Table(name="BreastCancerWithMeanDiff")
header_row_out = TableRow()
for h in headers + [f'mean_malignant_{top_feature}', f'mean_benign_{top_feature}', f'abs_difference_{top_feature}']:
    cell = TableCell()
    cell.addElement(P(text=h))
    header_row_out.addElement(cell)
table.addElement(header_row_out)
for rec in data:
    row = TableRow()
    for h in headers + [f'mean_malignant_{top_feature}', f'mean_benign_{top_feature}', f'abs_difference_{top_feature}']:
        cell = TableCell()
        cell.addElement(P(text=str(rec[h]) if h in rec and rec[h] is not None else ''))
        row.addElement(cell)
    table.addElement(row)
doc_out.spreadsheet.addElement(table)
doc_out.save("task1_breast_cancer_mean_diff.ods")
print("Task 1 file saved: task1_breast_cancer_mean_diff.ods")

Task 1 file saved: task1_breast_cancer_mean_diff.ods


## Task 1: Verifier

In [ ]:
# Task 1: Verifier
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P

def get_sheet_names(odf_doc):
    return [s.getAttribute("name") for s in odf_doc.getElementsByType(Table)]

def get_headers(sheet):
    header_row = sheet.getElementsByType(TableRow)[0]
    return [P.firstChild.data.strip() for P in header_row.getElementsByType(P) if P.firstChild]

def get_data_rows(sheet):
    return sheet.getElementsByType(TableRow)[1:]

def get_cell_text(cell):
    ps = cell.getElementsByType(P)
    return ps[0].firstChild.data.strip() if ps and ps[0].firstChild else ''

def validate_task1(task1_file, cleaned_file, top_feature):
    print("=== Task 1 Validation ===")
    cleaned_doc = load(cleaned_file)
    cleaned_sheet_names = get_sheet_names(cleaned_doc)
    print(f"- Original sheet names: {cleaned_sheet_names}")
    cleaned_sheet = cleaned_doc.getElementsByType(Table)[0]
    cleaned_headers = get_headers(cleaned_sheet)
    print(f"- Original headers: {cleaned_headers}")

    doc = load(task1_file)
    task1_sheet_names = get_sheet_names(doc)
    print(f"- Task1 sheet names: {task1_sheet_names}")
    table = [t for t in doc.getElementsByType(Table) if t.getAttribute("name") == "BreastCancerWithMeanDiff"]
    if not table:
        print("  Sheet 'BreastCancerWithMeanDiff' not found in output!")
        return
    table = table[0]
    headers = get_headers(table)
    print(f"- Task1 headers: {headers}")
    expected_headers = cleaned_headers + [f'mean_malignant_{top_feature}', f'mean_benign_{top_feature}', f'abs_difference_{top_feature}']
    header_names_match = headers == expected_headers
    print(f"- Header names match: {header_names_match}")
    print(f"- Header length match: {len(headers) == len(expected_headers)}")

    # Check values for the new columns
    vals_m = [float(get_cell_text(row.getElementsByType(TableCell)[headers.index(top_feature)])) for row in get_data_rows(table) if get_cell_text(row.getElementsByType(TableCell)[1]) == 'M']
    vals_b = [float(get_cell_text(row.getElementsByType(TableCell)[headers.index(top_feature)])) for row in get_data_rows(table) if get_cell_text(row.getElementsByType(TableCell)[1]) == 'B']
    mean_m = round(sum(vals_m)/len(vals_m), 3)
    mean_b = round(sum(vals_b)/len(vals_b), 3)
    abs_diff = round(abs(mean_m - mean_b), 3)
    value_errors = 0
    for row in get_data_rows(table):
        cells = row.getElementsByType(TableCell)
        try:
            mval = float(get_cell_text(cells[headers.index(f'mean_malignant_{top_feature}')]))
            bval = float(get_cell_text(cells[headers.index(f'mean_benign_{top_feature}')]))
            diffval = float(get_cell_text(cells[headers.index(f'abs_difference_{top_feature}')]))
            if abs(mval - mean_m) > 0.01 or abs(bval - mean_b) > 0.01 or abs(diffval - abs_diff) > 0.01:
                value_errors += 1
        except Exception:
            continue
    print(f"- Value validation errors: {value_errors}")

# Example usage:
validate_task1('task1_breast_cancer_mean_diff.ods', 'cleaned_breast_cancer.ods', 'area_worst')

=== Task 1 Validation ===
- Original sheet names: ['CleanedBreastCancer']
- Original headers: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst']
- Task1 sheet names: ['BreastCancerWithMeanDiff']
- Task1 headers: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se'

## Task 2: Find the Top 10 Patients with the Highest Value of the Most Discriminative Feature

In [ ]:
# Task 2: Find the Top 10 Patients with the Highest area_worst Value
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf.style import Style, TableCellProperties

# Use the top feature from Task 1
top_feature = "area_worst"

# 1. Find top 10 patients by the top feature
patient_feature = []
for rec in clean_data:
    try:
        pid = rec.get('id')
        val = float(rec.get(top_feature, 0) or 0)
        if pid and val > 0:
            patient_feature.append((pid, rec.get('diagnosis'), val, rec))
    except Exception:
        continue

top10 = sorted(patient_feature, key=lambda x: x[2], reverse=True)[:10]
top10_ids = {pid: rank+1 for rank, (pid, _, _, _) in enumerate(top10)}  # id: rank (1-based)

# 2. Add the top 10 rank to each record
enhanced_data2 = []
for rec in clean_data:
    pid = rec.get('id')
    rank = top10_ids.get(pid)
    enhanced_record = rec.copy()
    enhanced_record['top10_' + top_feature + '_rank'] = rank if rank is not None else ''
    enhanced_data2.append(enhanced_record)

# 3. Create ODS file with two sheets and color-coded cells
doc2 = OpenDocumentSpreadsheet()

# Define cell styles for color coding
green_cell_style = Style(name="GreenCell", family="table-cell")
green_cell_style.addElement(TableCellProperties(backgroundcolor="#CCFFCC"))
yellow_cell_style = Style(name="YellowCell", family="table-cell")
yellow_cell_style.addElement(TableCellProperties(backgroundcolor="#FFFF99"))
red_cell_style = Style(name="RedCell", family="table-cell")
red_cell_style.addElement(TableCellProperties(backgroundcolor="#FFCCCC"))
doc2.automaticstyles.addElement(green_cell_style)
doc2.automaticstyles.addElement(yellow_cell_style)
doc2.automaticstyles.addElement(red_cell_style)

# Sheet 1: Cleaned data + top 10 rank
table1 = Table(name="CleanedBreastCancerWithTop10Rank")
header_row1 = TableRow()
for h in clean_headers + ['top10_' + top_feature + '_rank']:
    cell = TableCell()
    cell.addElement(P(text=h))
    header_row1.addElement(cell)
table1.addElement(header_row1)
for rec in enhanced_data2:
    row = TableRow()
    for h in clean_headers:
        val = rec.get(h, '')
        cell = TableCell()
        cell.addElement(P(text=str(val) if val is not None else ''))
        row.addElement(cell)
    # Color code the top10 rank cell
    rank = rec.get('top10_' + top_feature + '_rank')
    style = None
    if str(rank) in ['1', '2', '3']:
        style = "GreenCell"
    elif str(rank) in ['4', '5', '6', '7']:
        style = "YellowCell"
    elif str(rank) in ['8', '9', '10']:
        style = "RedCell"
    cell = TableCell(stylename=style) if style else TableCell()
    cell.addElement(P(text=str(rank) if rank is not None else ''))
    row.addElement(cell)
    table1.addElement(row)
doc2.spreadsheet.addElement(table1)

# Sheet 2: Top 10 patients by the feature, color-coded
table2 = Table(name="Top10By" + top_feature)
header_row2 = TableRow()
for col in ['Rank', 'Patient ID', 'Diagnosis', top_feature]:
    cell = TableCell()
    cell.addElement(P(text=col))
    header_row2.addElement(cell)
table2.addElement(header_row2)
for idx, (pid, diagnosis, val, _) in enumerate(top10):
    if idx < 3:
        cell_style = "GreenCell"
    elif idx < 7:
        cell_style = "YellowCell"
    else:
        cell_style = "RedCell"
    trow = TableRow()
    for value in [str(idx+1), str(pid), str(diagnosis), str(val)]:
        cell = TableCell(stylename=cell_style)
        cell.addElement(P(text=value))
        trow.addElement(cell)
    table2.addElement(trow)
doc2.spreadsheet.addElement(table2)

task2_file = f'task2_top10_{top_feature}.ods'
doc2.save(task2_file)
print(f'Task 2 file saved: {task2_file}')

Task 2 file saved: task2_top10_area_worst.ods


## Task 2: Verifier

In [ ]:
# Task 2: Verifier
from odf.style import TableCellProperties

def get_cell_style(cell):
    return cell.getAttribute("stylename")

def get_style_color_map(odf_doc):
    color_map = {}
    for style in odf_doc.automaticstyles.childNodes:
        if getattr(style, "tagName", None) == "style:style":
            props = style.getElementsByType(TableCellProperties)
            if props and props[0].getAttribute("backgroundcolor"):
                color_map[style.getAttribute("name")] = props[0].getAttribute("backgroundcolor")
    return color_map

def validate_task2(task2_file, cleaned_file, top_feature):
    print("=== Task 2 Validation ===")
    cleaned_doc = load(cleaned_file)
    cleaned_sheet_names = get_sheet_names(cleaned_doc)
    print(f"- Original sheet names: {cleaned_sheet_names}")
    cleaned_sheet = cleaned_doc.getElementsByType(Table)[0]
    cleaned_headers = get_headers(cleaned_sheet)
    print(f"- Original headers: {cleaned_headers}")

    doc = load(task2_file)
    task2_sheet_names = get_sheet_names(doc)
    print(f"- Task2 sheet names: {task2_sheet_names}")
    # Sheet 2: Top 10 patients
    table = [t for t in doc.getElementsByType(Table) if t.getAttribute("name") == "Top10By" + top_feature]
    if not table:
        print(f"  Sheet 'Top10By{top_feature}' not found in output!")
        return
    table = table[0]
    headers = get_headers(table)
    print(f"- Task2 headers: {headers}")
    header_names_match = headers == ['Rank', 'Patient ID', 'Diagnosis', top_feature]
    print(f"- Header names match: {header_names_match}")
    print(f"- Header length match: {len(headers) == 4}")

    # Recompute top 10 by area_worst from cleaned data
    patient_feature = []
    for row in get_data_rows(cleaned_sheet):
        cells = row.getElementsByType(TableCell)
        row_dict = dict(zip(cleaned_headers, [get_cell_text(cell) for cell in cells]))
        try:
            pid = row_dict.get('id')
            val = float(row_dict.get(top_feature, 0) or 0)
            if pid and val > 0:
                patient_feature.append((pid, row_dict.get('diagnosis'), val))
        except Exception:
            continue
    expected = sorted(patient_feature, key=lambda x: x[2], reverse=True)[:10]
    color_map = get_style_color_map(doc)
    value_errors = 0
    color_errors = 0
    for i, row in enumerate(get_data_rows(table)):
        cells = row.getElementsByType(TableCell)
        values = [get_cell_text(cell) for cell in cells]
        exp_rank = str(i+1)
        exp_pid, exp_diag, exp_val = expected[i]
        # Value check
        if values[0] != exp_rank or values[1] != exp_pid or values[2] != exp_diag or abs(float(values[3]) - exp_val) > 0.01:
            print(f'  Row {i+1}: value mismatch (expected {exp_rank}, {exp_pid}, {exp_diag}, {exp_val}, got {values})')
            value_errors += 1
        # Color check (all cells in row should have the same color)
        for cell in cells:
            style = get_cell_style(cell)
            actual_color = color_map.get(style)
            if i < 3:
                expected_color = "#CCFFCC"
            elif i < 7:
                expected_color = "#FFFF99"
            else:
                expected_color = "#FFCCCC"
            if actual_color != expected_color:
                print(f'  Row {i+1}: color mismatch (expected {expected_color}, got {actual_color})')
                color_errors += 1
    print(f"- Value validation errors: {value_errors}")
    print(f"- Color validation errors: {color_errors}")

# Example usage:
validate_task2('task2_top10_area_worst.ods', 'cleaned_breast_cancer.ods', 'area_worst')

=== Task 2 Validation ===
- Original sheet names: ['CleanedBreastCancer']
- Original headers: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst']
- Task2 sheet names: ['CleanedBreastCancerWithTop10Rank', 'Top10Byarea_worst']
- Task2 headers: ['Rank', 'Patient ID', 'Diagnosis', 'area_worst']
- Header names match: True
- Header length match: True
- Value validation errors: 0
- Color validation errors: 0
